In [2]:
import pandas as pd
ruta = "../data/archive/"
orders = pd.read_csv("/Users/pma/Bootcamp/Proyecto-ML---Olist-Delivery-/data/archive/olist_orders_dataset.csv")
items = pd.read_csv("/Users/pma/Bootcamp/Proyecto-ML---Olist-Delivery-/data/archive/olist_order_items_dataset.csv")

orders.info()
items.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB
<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 no

In [ ]:
# Fechas a datetime
cols_fecha = ["order_purchase_timestamp", "order_approved_at",
              "order_delivered_carrier_date", "order_delivered_customer_date",
              "order_estimated_delivery_date"] # Paso de str a datetime 
for c in cols_fecha:
    orders[c] = pd.to_datetime(orders[c])

# Solo pedidos entregados y con fecha de entrega
entregados = orders[(orders["order_status"] == "delivered")
                    & orders["order_delivered_customer_date"].notna()].copy() # Elijo los pedidos que fueron entregados y los que no estan nulos.

# Target: 1 si llegó después de la fecha estimada
entregados["tarde"] = (entregados["order_delivered_customer_date"] # creo la columna "tarde", 1 si llego a tiempo, 0 si no llego a tiempo, de aca el astype. 
                       > entregados["order_estimated_delivery_date"]).astype(int) # me fijo cuales son los pedidos que llegaron tarde de acuerdo a la fecha estimada.

print(entregados["tarde"].value_counts(normalize=True)) # Me da la proporcion de cada clase (0 y 1)

# Items resumidos a una fila por pedido
items_por_pedido = items.groupby("order_id").agg( 
    n_items=("order_item_id", "count"),
    precio_total=("price", "sum"),
    flete_total=("freight_value", "sum"),
    n_vendedores=("seller_id", "nunique"), # columna de items y que calcular. 
).reset_index() 

orders["order_status"].value_counts()



tarde
0    0.918876
1    0.081124
Name: proportion, dtype: float64


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64